In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.config import DEFAULT_STGCN_PARAMS_V1, POSE_DATASET_ROOT, DATASET_ROOT
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
import skelbumentations as S
import torch

train_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train", max_people=3)


In [21]:
def augment_pose(tensor):
    """
    tensor shape: [3, T, V, M]
    channels: x, y, confidence
    """
    opposite_coco_points = [
        [5, 6],  # left shoulder, right shoulder
        [7, 8],  # left elbow, right elbow
        [9, 10], # left wrist, right wrist
        [11, 12], # left hip, right hip
        [13, 14], # left knee, right knee
        [15, 16], # left ankle, right ankle
    ]

    pipeline = S.Compose([
        S.SelectRandomFrames(
            [S.WholeOcclusion()], 
            min_num=25, 
            max_num=50, 
            p=1.0, 
        ), 
        S.SelectRandomFrames(
            [S.MirrorPerturbation(opposite_coco_points)], 
            min_num=1, 
            max_num=4, 
            p=1.0,
        )
    ])

    tensor = tensor.clone()
    C, T, V, M = tensor.shape

    for person_index in range(M):
        person_pose = tensor[:, :, :, person_index] # [3, T, V]

        if person_pose[2].sum() == 0: # continue if no person exists
            continue

        # pose sequence has to be numpy array with format (T, V, C)
        keypoints = person_pose.permute(1, 2, 0).cpu().numpy().copy()

        # Skelbumentations expects a mask indicating joints that are already missing
        # confidence 0 means keypoint was already missing 
        invalid = (person_pose[2] == 0).cpu().numpy().copy()
        augmented = pipeline(keypoints=keypoints, invalid=invalid)

        augmented_person_pose = torch.from_numpy(augmented["keypoints"]).permute(2, 0, 1)
        tensor[:, :, :, person_index] = augmented_person_pose

    return tensor


In [22]:
tensor, label = train_dataset[40]

augmented_tensor = augment_pose(tensor)

print("Original shape: ", tensor.shape)
print("Augmented shape:", augmented_tensor.shape)

print("Tensor changed:",
      not torch.equal(tensor, augmented_tensor))

difference = tensor != augmented_tensor

changed_frames = difference.any(dim=0).any(dim=1)

print("Changed frames:")
print(torch.where(changed_frames)[0].tolist())

Original shape:  torch.Size([3, 150, 17, 3])
Augmented shape: torch.Size([3, 150, 17, 3])
Tensor changed: True
Changed frames:
[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 26, 27, 27, 28, 28, 29, 29, 30, 30, 31, 31, 32, 32, 33, 33, 34, 34, 35, 35, 36, 36, 37, 37, 38, 38, 39, 39, 40, 40, 41, 41, 42, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 125, 126, 127, 143, 144]


In [23]:
for person in range(tensor.shape[-1]):
    person_changed = not torch.equal(
        tensor[:, :, :, person],
        augmented_tensor[:, :, :, person],
    )

    print(f"Person {person}: {person_changed}")

Person 0: True
Person 1: True
Person 2: False


In [24]:
print(
    augmented_tensor[:2].min().item(),
    augmented_tensor[:2].max().item(),
)

0.0 1.0
